# Character TF-IDF — Quick Search (~10 min)

Lightweight char n-gram TF-IDF baseline on the DVC split.
**12 runs** (2 preprocessing × 2 n-gram ranges × 3 fast models). No XGBoost/RF/trees, no test
features during grid, smaller vocabulary.

In [ ]:
import json
import re
import warnings
from datetime import datetime
from itertools import product
from pathlib import Path
from time import perf_counter

import joblib
import mlflow
import pandas as pd
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

from helpers.data import load_processed_data, load_sample_submission
from helpers.metrics import compute_all_metrics
from helpers.mlflow import log_common_context, log_metrics, log_model_artifacts, start_notebook_run
from helpers.submission import build_submission, save_submission

warnings.filterwarnings('ignore', category=FutureWarning)

MAX_CHAR_FEATURES = 8_000
CHAR_NGRAM_RANGES = [(2, 4), (2, 5)]
PREPROCESS_TRACKS = ['track_a_raw', 'track_b_normalized']
MODEL_FAMILIES = ['logistic_regression', 'linear_svc', 'naive_bayes']


def log_step(message: str) -> None:
    print(f'[{datetime.now():%H:%M:%S}] {message}')

## Load Data

In [ ]:
PROCESSED_DIR = Path('../data/processed')
RAW_DIR = Path('../data/raw')
KAGGLE_INPUT_DIR = Path('/kaggle/input/competitions/contradictory-my-dear-watson')

if (PROCESSED_DIR / 'train_split.csv').exists():
    train_df, val_df, test_df = load_processed_data(PROCESSED_DIR)
    sample_submission = load_sample_submission(RAW_DIR / 'sample_submission.csv')
    data_source = 'dvc_processed_split'
elif (KAGGLE_INPUT_DIR / 'train.csv').exists():
    full_train_df = pd.read_csv(KAGGLE_INPUT_DIR / 'train.csv')
    test_df = pd.read_csv(KAGGLE_INPUT_DIR / 'test.csv')
    sample_submission = pd.read_csv(KAGGLE_INPUT_DIR / 'sample_submission.csv')
    train_df, val_df = train_test_split(
        full_train_df,
        test_size=0.2,
        random_state=42,
        stratify=full_train_df['label'],
    )
    data_source = 'kaggle_input_fallback_split'
else:
    raise FileNotFoundError('Could not find DVC processed splits or Kaggle input files.')

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)
y_train = train_df['label'].astype(int)

log_step(f'Train={len(train_df)}, Val={len(val_df)}, Test={len(test_df)}, source={data_source}')

## Preprocessing + Char TF-IDF

- **Track A:** whitespace only
- **Track B:** lowercase + strip punctuation
- **Stop words:** `keep_all` (char n-grams rarely benefit from word-level stop removal)

Features: premise + hypothesis only (no overlap/difference blocks — faster, less memory).

In [ ]:
PUNCTUATION_PATTERN = re.compile(r"[^\w\s]", re.UNICODE)


def preprocess_series(series: pd.Series, track: str) -> pd.Series:
    texts = series.fillna('').astype(str)
    if track == 'track_a_raw':
        return texts.str.split().str.join(' ')
    normalized = texts.str.lower().str.replace(PUNCTUATION_PATTERN, ' ', regex=True)
    return normalized.str.split().str.join(' ')


def fit_char_vectorizer(train_frame, feature_config):
    track = feature_config['preprocess_track']
    vectorizer = TfidfVectorizer(
        analyzer='char',
        ngram_range=feature_config['ngram_range'],
        max_features=MAX_CHAR_FEATURES,
        lowercase=False,
        sublinear_tf=True,
    )
    corpus = pd.concat([
        preprocess_series(train_frame['premise'], track),
        preprocess_series(train_frame['hypothesis'], track),
    ])
    vectorizer.fit(corpus)
    return vectorizer


def make_char_features(frame, vectorizer, feature_config):
    track = feature_config['preprocess_track']
    premise_vec = vectorizer.transform(preprocess_series(frame['premise'], track))
    hypothesis_vec = vectorizer.transform(preprocess_series(frame['hypothesis'], track))
    return hstack([premise_vec, hypothesis_vec], format='csr')


def build_train_val_features(train_frame, val_frame, feature_config):
    vectorizer = fit_char_vectorizer(train_frame, feature_config)
    return (
        make_char_features(train_frame, vectorizer, feature_config),
        make_char_features(val_frame, vectorizer, feature_config),
        vectorizer,
    )


def make_classifier_pipeline(model_family: str) -> Pipeline:
    estimators = {
        'logistic_regression': LogisticRegression(
            C=1.0,
            max_iter=1000,
            random_state=42,
        ),
        'linear_svc': LinearSVC(random_state=42),
        'naive_bayes': MultinomialNB(),
    }
    return Pipeline([('clf', estimators[model_family])])

## Grid Search (12 runs)

In [ ]:
FEATURE_CONFIGS = [
    {
        'config_id': f'{track}__chars_{ngram[0]}_{ngram[1]}',
        'preprocess_track': track,
        'stop_words': 'keep_all',
        'ngram_range': ngram,
    }
    for track, ngram in product(PREPROCESS_TRACKS, CHAR_NGRAM_RANGES)
]

SEARCH_PLAN = list(product(FEATURE_CONFIGS, MODEL_FAMILIES))
log_step(f'{len(SEARCH_PLAN)} runs planned')

feature_cache = {}
grid_results = []
grid_started = perf_counter()

for feature_config, model_family in SEARCH_PLAN:
    config_id = feature_config['config_id']
    run_label = f'{config_id}__{model_family}'
    run_started = perf_counter()

    if config_id not in feature_cache:
        t0 = perf_counter()
        x_train, x_val, vectorizer = build_train_val_features(train_df, val_df, feature_config)
        feature_cache[config_id] = (x_train, x_val, vectorizer)
        log_step(f'Features {config_id} in {perf_counter() - t0:.1f}s')

    x_train, x_val, _ = feature_cache[config_id]
    pipeline = make_classifier_pipeline(model_family)
    pipeline.fit(x_train, y_train)
    val_preds = pipeline.predict(x_val)
    metrics = compute_all_metrics(val_df, val_preds)
    elapsed = perf_counter() - run_started

    grid_results.append({
        'run_label': run_label,
        'config_id': config_id,
        'preprocess_track': feature_config['preprocess_track'],
        'ngram_range': str(feature_config['ngram_range']),
        'model_family': model_family,
        'accuracy': metrics['accuracy'],
        'f1_macro': metrics['f1_macro'],
        'elapsed_seconds': round(elapsed, 1),
    })
    log_step(
        f'{run_label} acc={metrics["accuracy"]:.4f} f1={metrics["f1_macro"]:.4f} ({elapsed:.1f}s)'
    )

log_step(f'Grid done in {(perf_counter() - grid_started) / 60:.1f} min')
results_df = pd.DataFrame(grid_results).sort_values(['f1_macro', 'accuracy'], ascending=False)
results_df

## Production (best row + submission)

In [ ]:
best_result = results_df.iloc[0]
best_config_id = best_result['config_id']
best_model_family = best_result['model_family']
best_feature_config = next(c for c in FEATURE_CONFIGS if c['config_id'] == best_config_id)

log_step(f'Best: {best_result["run_label"]} | f1={best_result["f1_macro"]:.4f}')

vectorizer = fit_char_vectorizer(train_df, best_feature_config)
x_train = make_char_features(train_df, vectorizer, best_feature_config)
x_val = make_char_features(val_df, vectorizer, best_feature_config)
x_test = make_char_features(test_df, vectorizer, best_feature_config)

production_pipeline = make_classifier_pipeline(best_model_family)
production_pipeline.fit(x_train, y_train)

production_metrics = compute_all_metrics(val_df, production_pipeline.predict(x_val))
test_preds = production_pipeline.predict(x_test)

RUN_NAME = f'char_tfidf_quick_{best_config_id}__{best_model_family}'
submission_path = Path('../submissions') / f'{RUN_NAME}.csv'
save_submission(build_submission(sample_submission, test_preds), submission_path)

results_path = Path('../submissions') / 'char_tfidf_quick_results.csv'
results_df.to_csv(results_path, index=False)

model_path = Path('../submissions') / f'{RUN_NAME}_pipeline.joblib'
vectorizer_path = Path('../submissions') / f'{RUN_NAME}_vectorizer.joblib'
joblib.dump(production_pipeline, model_path)
joblib.dump(vectorizer, vectorizer_path)

serving_type = 'sklearn_char_tfidf_pair_features'
metadata = {
    'run_name': RUN_NAME,
    'data_source': data_source,
    'serving_type': serving_type,
    'feature_config': {**best_feature_config, 'ngram_range': list(best_feature_config['ngram_range'])},
    'model_family': best_model_family,
    'grid_runs': len(results_df),
    'best_grid_result': best_result.to_dict(),
}

with start_notebook_run(RUN_NAME, tags={'features': 'char_tfidf_quick', 'stage': 'production_candidate'}):
    mlflow.log_params({
        'model_family': best_model_family,
        'config_id': best_config_id,
        'max_char_features': MAX_CHAR_FEATURES,
        'data_source': data_source,
    })
    log_metrics(production_metrics)
    log_common_context('../configs/data_split.yaml', '../data/processed/split_metadata.json')
    mlflow.log_artifact(str(submission_path), artifact_path='submissions')
    log_model_artifacts(
        artifacts={'pipeline.joblib': model_path, 'vectorizer.joblib': vectorizer_path},
        metadata=metadata,
        artifact_path='model',
    )

pd.Series({
    'run_name': RUN_NAME,
    'accuracy': production_metrics['accuracy'],
    'f1_macro': production_metrics['f1_macro'],
    'submission_path': str(submission_path),
})